# UK Biobank Synthetic Dataset Tutorial

This notebook provides a comprehensive guide to downloading and working with the UK Biobank Synthetic Dataset.

## What is the UK Biobank Synthetic Dataset?

The UK Biobank Synthetic Dataset is a **synthetic (fake) version** of the real UK Biobank dataset. It's designed for:
- Testing software systems that need to handle large healthcare datasets
- Practicing data processing pipelines without privacy concerns
- System performance testing

**Important**: The data is randomly generated and **not internally consistent**. For example, it might contain:
- Prostate cancer in female participants
- Medical events after death
- Dates without corresponding diagnoses

**Reference**: [UK Biobank Synthetic Dataset](https://biobank.ndph.ox.ac.uk/synthetic_dataset/)

## Dataset Overview

The dataset contains 4 main categories:

1. **Tabular Records** (23 TSV files) - Main phenotype data: ~600K participants × ~27K columns
2. **Medical Records** (6 text files) - GP clinical records: ~400M rows
3. **Genetic Records** (dictionary + 26 chromosome files) - SNP genotype data: ~600K participants × 840K SNPs
4. **Bulk Files** (37 zip archives) - ~6M files for system testing only

## Prerequisites

- Python environment with required packages (`requests`, `polars`)
- Internet connection (for downloading files)
- Sufficient disk space (tabular data is several GB, genetic data is much larger)

In [ ]:
%load_ext autoreload
%autoreload 2

# Import the UK Biobank Synthetic Dataset module
import sys
import os
from pathlib import Path

# Add parent directory to path if package not installed
if 'synthlab' not in sys.modules:
    sys.path.insert(0, os.path.abspath(os.path.join(os.getcwd(), '..')))

from synthlab import (
    list_available_files,
    download_file,
    download_category,
    load_tabular_data,
    load_medical_records,
    load_genetic_dictionary,
    get_cache_dir,
)

print("✓ UK Biobank Synthetic Dataset module imported successfully")
print(f"\nDefault cache directory: {get_cache_dir()}")
print("Data will be saved to ~/.cache/synthlab/ukbiobank_synthetic/ by default")

✓ UK Biobank Synthetic Dataset module imported successfully


## 1. Exploring Available Files

First, let's see what files are available in each category.

In [2]:
# List all available files by category
print("Available files in UK Biobank Synthetic Dataset:")
print("=" * 60)

available = list_available_files()

for category, files in available.items():
    print(f"\n{category.upper()} ({len(files)} files):")
    print("-" * 60)
    # Show first 8 files as examples
    for filename in files[:8]:
        print(f"  • {filename}")
    if len(files) > 8:
        print(f"  ... and {len(files) - 8} more files")

Available files in UK Biobank Synthetic Dataset:

TABULAR (24 files):
------------------------------------------------------------
  • 41257_HES_SimDates.tsv
  • 41260_HES_SimDates.tsv
  • 41262_HES_SimDates.tsv
  • 41263_HES_SimDates.tsv
  • 41280_HES_SimDates.tsv
  • 41281_HES_SimDates.tsv
  • 41282_HES_SimDates.tsv
  • 41283_HES_SimDates.tsv
  ... and 16 more files

MEDICAL (7 files):
------------------------------------------------------------
  • set3a1.txt
  • set3a2a.txt
  • set3a3.txt
  • set3a4.txt
  • set3b.txt
  • set3c.txt
  • medrec.md5

GENETIC (29 files):
------------------------------------------------------------
  • gene_dic.dat
  • rand_chr1.dat.gz
  • rand_chr2.dat.gz
  • rand_chr3.dat.gz
  • rand_chr4.dat.gz
  • rand_chr5.dat.gz
  • rand_chr6.dat.gz
  • rand_chr7.dat.gz
  ... and 21 more files

BULK (37 files):
------------------------------------------------------------
  • bulk_20158.zip
  • bulk_20203.zip
  • bulk_20205.zip
  • bulk_20206.zip
  • bulk_20220.zip


### 1.1 List Files in a Specific Category

You can also list files for a specific category:

In [3]:
# List only tabular files (most commonly used)
tabular_files = list_available_files(category="tabular")
print("Tabular files:")
for filename in tabular_files["tabular"]:
    print(f"  • {filename}")

Tabular files:
  • 41257_HES_SimDates.tsv
  • 41260_HES_SimDates.tsv
  • 41262_HES_SimDates.tsv
  • 41263_HES_SimDates.tsv
  • 41280_HES_SimDates.tsv
  • 41281_HES_SimDates.tsv
  • 41282_HES_SimDates.tsv
  • 41283_HES_SimDates.tsv
  • bulk_strings.tsv
  • dates_death.tsv
  • datetime_fields_2.tsv
  • datetime_fields.tsv
  • fo_fields_trimmed.tsv
  • integer_arrays_part1.tsv
  • integer_arrays_part2.tsv
  • integer_diet_quest_fields.tsv
  • integer_no_arrays.tsv
  • integer_other_quest_fields.tsv
  • oaa_fields.tsv
  • real_fields1.tsv
  • real_fields2.tsv
  • string_fields1.tsv
  • string_fields2.tsv
  • tabular.md5


## 2. Downloading Data

### 2.1 Download Tabular Data (Recommended)

The **tabular data** is the most commonly used category. It contains phenotype data (survey responses, measurements, clinical data) for ~600,000 participants.

**Note**: The download may take a while depending on your internet connection. Files are automatically verified using MD5 checksums.

In [ ]:
# Download tabular data
# This will download all 23 TSV files (~several GB)
# By default, files are saved to ~/.cache/synthlab/ukbiobank_synthetic/tabular/
# Uncomment to actually download:
tabular_dir = download_category(
    "tabular",
    verify_md5=True,  # Verify file integrity
    overwrite=False    # Skip files that already exist
)
# # Or specify a custom directory:
# # tabular_dir = download_category("tabular", Path("/custom/path"))

print("To download tabular data, uncomment the code above.")
print(f"Files will be saved to: {get_cache_dir() / 'tabular'}")
print("\nNote: This is a large download. Tabular data contains:")
print("  • 23 TSV files")
print("  • ~27,000 columns × 600,000 rows")
print("  • Several GB of data")


Output directory: /home/schilder/.cache/synthlab/ukbiobank_synthetic/tabular
  ↓ 41257_HES_SimDates.tsv... ✓ (verified)
  ↓ 41260_HES_SimDates.tsv... ✓ (verified)
  ↓ 41262_HES_SimDates.tsv... ✓ (verified)
  ↓ 41263_HES_SimDates.tsv... ✓ (verified)
  ↓ 41280_HES_SimDates.tsv... 

### 2.2 Download a Single File

If you only need specific files, you can download them individually:

In [ ]:
# Example: Download just the death dates file
# By default, files are saved to ~/.cache/synthlab/ukbiobank_synthetic/{category}/
# Uncomment to download:
# death_file = download_file(
#     "dates_death.tsv",
#     category="tabular",
#     verify_md5=True
# )
# print(f"Downloaded: {death_file}")

print("To download a single file, uncomment the code above.")
print(f"Files will be saved to: {get_cache_dir() / 'tabular'}")
print("\nExample files you might want to download individually:")
print("  • dates_death.tsv - Death dates")
print("  • integer_no_arrays.tsv - Integer survey responses")
print("  • real_fields1.tsv - Real-valued measurements (BMI, blood pressure, etc.)")
print("  • string_fields1.tsv - Text responses")

### 2.3 Download Other Categories

**Medical Records**: GP clinical records (~400M rows) - Very large, only download if needed.

**Genetic Records**: SNP genotype data - Extremely large (each chromosome file is several GB compressed).

**Bulk Files**: For system testing only, not for analysis.

In [ ]:
# Example: Download medical records (uncomment to use)
# WARNING: This is very large (~400M rows)
# By default, saved to ~/.cache/synthlab/ukbiobank_synthetic/medical/
# medical_dir = download_category(
#     "medical",
#     verify_md5=True
# )

# Example: Download genetic data (uncomment to use)
# WARNING: This is extremely large (each chromosome file is several GB)
# By default, saved to ~/.cache/synthlab/ukbiobank_synthetic/genetic/
# genetic_dir = download_category(
#     "genetic",
#     verify_md5=True
# )

print("⚠️  Medical and genetic data are very large.")
print("Only download if you specifically need:")
print("  • Medical records: GP clinical diagnosis codes and visit data")
print("  • Genetic data: SNP genotypes for genetic association studies")
print(f"\nDefault cache locations:")
print(f"  • Medical: {get_cache_dir() / 'medical'}")
print(f"  • Genetic: {get_cache_dir() / 'genetic'}")

## 3. Loading and Exploring Tabular Data

Once you've downloaded the tabular data, you can load it into Polars DataFrames for analysis.

In [ ]:
# Load all tabular data files
# By default, loads from ~/.cache/synthlab/ukbiobank_synthetic/tabular/
tabular_dir = get_cache_dir() / "tabular"

if tabular_dir.exists() and any(tabular_dir.glob("*.tsv")):
    print("Loading tabular data...")
    print("=" * 60)
    
    # Load all files (use sample_rows to limit for testing)
    # Remove sample_rows parameter to load full dataset
    tabular_data = load_tabular_data(
        tabular_dir,
        sample_rows=1000  # Load first 1000 rows for demo (remove for full data)
    )
    
    print(f"\n✓ Loaded {len(tabular_data)} tabular files")
    print("\nFile summary:")
    for name, df in tabular_data.items():
        print(f"  • {name}: {len(df)} rows × {len(df.columns)} columns")
else:
    print(f"⚠️  Tabular data directory not found: {tabular_dir}")
    print("   Please download the data first (Section 2.1)")
    tabular_data = {}

### 3.1 Explore Individual Files

Let's look at the structure of some key files:

In [ ]:
if tabular_data:
    # Show structure of death dates file
    if "dates_death" in tabular_data:
        print("Death Dates File:")
        print("=" * 60)
        df = tabular_data["dates_death"]
        print(f"Shape: {len(df)} rows × {len(df.columns)} columns")
        print(f"\nColumns: {', '.join(df.columns[:10])}...")
        print(f"\nFirst few rows:")
        print(df.head())
        
    # Show structure of integer fields
    if "integer_no_arrays" in tabular_data:
        print("\n\nInteger Fields File:")
        print("=" * 60)
        df = tabular_data["integer_no_arrays"]
        print(f"Shape: {len(df)} rows × {len(df.columns)} columns")
        print(f"\nFirst few columns: {', '.join(df.columns[:10])}...")
        print(f"\nFirst few rows (first 5 columns):")
        print(df.select(df.columns[:5]).head())
        
    # Show structure of real-valued fields
    if "real_fields1" in tabular_data:
        print("\n\nReal-Valued Fields File (measurements like BMI, blood pressure):")
        print("=" * 60)
        df = tabular_data["real_fields1"]
        print(f"Shape: {len(df)} rows × {len(df.columns)} columns")
        print(f"\nFirst few columns: {', '.join(df.columns[:10])}...")
        print(f"\nFirst few rows (first 5 columns):")
        print(df.select(df.columns[:5]).head())
else:
    print("No tabular data loaded. Please download and load data first.")

### 3.2 Load Specific Files

You can also load specific files using a pattern:

In [ ]:
if tabular_dir.exists():
    # Load only integer-related files
    integer_data = load_tabular_data(
        tabular_dir,
        file_pattern="integer",
        sample_rows=1000
    )
    
    print(f"Loaded {len(integer_data)} integer-related files:")
    for name in integer_data.keys():
        print(f"  • {name}")
else:
    print("Tabular directory not found. Download data first.")

## 4. Basic Data Analysis Examples

Let's perform some basic analyses on the loaded data.

In [ ]:
import polars as pl

if tabular_data:
    # Example 1: Check participant count
    if "dates_death" in tabular_data:
        df = tabular_data["dates_death"]
        n_participants = df["EID"].n_unique()
        print(f"Total participants: {n_participants:,}")
        
        # Count deaths
        if "40000.0.0" in df.columns:  # Field 40000 is typically death date
            deaths = df.filter(pl.col("40000.0.0").is_not_null())
            print(f"Participants with death dates: {len(deaths):,}")
    
    # Example 2: Basic statistics on real-valued measurements
    if "real_fields1" in tabular_data:
        print("\n" + "=" * 60)
        print("Real-Valued Measurements Summary:")
        print("=" * 60)
        df = tabular_data["real_fields1"]
        
        # Get numeric columns (excluding EID)
        numeric_cols = [col for col in df.columns if col != "EID"]
        
        if numeric_cols:
            # Calculate basic statistics for first few numeric columns
            stats = df.select([
                pl.col(col).mean().alias(f"{col}_mean")
                for col in numeric_cols[:5]
            ])
            print("\nMean values for first 5 measurement columns:")
            print(stats)
else:
    print("No data loaded. Please download and load data first.")

### 4.1 Merge Data from Multiple Files

You can join data from different files using the EID (participant identifier):

In [ ]:
if tabular_data and len(tabular_data) >= 2:
    # Example: Merge death dates with another file
    if "dates_death" in tabular_data and "integer_no_arrays" in tabular_data:
        death_df = tabular_data["dates_death"]
        integer_df = tabular_data["integer_no_arrays"]
        
        # Join on EID
        merged = death_df.join(
            integer_df.select(["EID"] + integer_df.columns[1:6]),  # Select EID + first 5 other columns
            on="EID",
            how="left"
        )
        
        print("Merged death dates with integer fields:")
        print(f"  Shape: {len(merged)} rows × {len(merged.columns)} columns")
        print(f"  Columns: {', '.join(merged.columns[:8])}...")
        print(f"\nFirst few rows:")
        print(merged.head())
else:
    print("Need at least 2 data files loaded to demonstrate merging.")

## 5. Working with Medical Records

Medical records contain GP clinical data with diagnosis codes (Read codes) and visit information.

In [ ]:
# Load medical records (if downloaded)
# By default, loads from ~/.cache/synthlab/ukbiobank_synthetic/medical/
medical_dir = get_cache_dir() / "medical"

if medical_dir.exists() and any(medical_dir.glob("*.txt")):
    print("Loading medical records...")
    print("=" * 60)
    print("⚠️  Note: Medical records are very large (~400M rows)")
    print("   Using sample_rows to limit for demonstration\n")
    
    # Load a sample (remove sample_rows for full dataset)
    medical_records = load_medical_records(
        medical_dir,
        sample_rows=10000  # Load first 10,000 rows for demo
    )
    
    print(f"\n✓ Loaded {len(medical_records):,} medical records")
    print(f"\nColumns: {', '.join(medical_records.columns)}")
    print(f"\nFirst few records:")
    print(medical_records.head())
    
    # Example: Count records per participant
    records_per_participant = medical_records.group_by("EID").agg(
        pl.count().alias("n_records")
    )
    print(f"\nRecords per participant (sample):")
    print(records_per_participant.head(10))
else:
    print(f"⚠️  Medical records directory not found: {medical_dir}")
    print("   To download medical records:")
    print("   download_category('medical', medical_dir)")
    print("\n   Note: Medical records are very large (~400M rows)")

## 6. Working with Genetic Data

Genetic data includes SNP genotype information. The dictionary file maps SNP IDs to variants, and chromosome files contain genotype data.

In [ ]:
# Load genetic dictionary (if downloaded)
# By default, loads from ~/.cache/synthlab/ukbiobank_synthetic/genetic/
genetic_dir = get_cache_dir() / "genetic"

if genetic_dir.exists() and (genetic_dir / "gene_dic.dat").exists():
    print("Loading genetic dictionary...")
    print("=" * 60)
    
    gene_dict = load_genetic_dictionary(genetic_dir)
    
    print(f"\n✓ Loaded dictionary with {len(gene_dict):,} SNPs")
    print(f"\nColumns: {', '.join(gene_dict.columns)}")
    print(f"\nFirst few SNPs:")
    print(gene_dict.head())
    
    # Example: Count SNPs per chromosome
    snps_per_chr = gene_dict.group_by("chromosome").agg(
        pl.count().alias("n_snps")
    ).sort("chromosome")
    print(f"\nSNPs per chromosome:")
    print(snps_per_chr)
    
    print("\n⚠️  Note: Chromosome data files (rand_chr*.dat.gz) are very large.")
    print("   They need to be decompressed and parsed separately.")
    print("   Each file contains genotypes for all participants for that chromosome.")
else:
    print(f"⚠️  Genetic dictionary not found: {genetic_dir / 'gene_dic.dat'}")
    print("   To download genetic data:")
    print("   download_category('genetic', genetic_dir)")
    print("\n   Note: Genetic data is extremely large (each chromosome file is several GB)")

## 7. Understanding Field Names

UK Biobank uses a specific naming convention: `FieldID.InstanceID.ArrayID`

- **FieldID**: The data field number (e.g., 31 = sex, 21001 = BMI)
- **InstanceID**: Visit/assessment instance (0 = baseline, 1 = first repeat, etc.)
- **ArrayID**: Array index for fields with multiple values

You can look up field definitions at: http://biobank.ndph.ox.ac.uk/showcase/schema.cgi

In [ ]:
# Example: Identify common fields
if tabular_data:
    # Look for common field IDs in column names
    print("Common UK Biobank Fields:")
    print("=" * 60)
    
    common_fields = {
        "31": "Sex",
        "34": "Year of birth",
        "52": "Month of birth",
        "21001": "Body mass index (BMI)",
        "50": "Standing height",
        "4080": "Systolic blood pressure",
        "4079": "Diastolic blood pressure",
        "40000": "Date of death",
    }
    
    for field_id, description in common_fields.items():
        # Search for this field in all loaded dataframes
        found = False
        for name, df in tabular_data.items():
            matching_cols = [col for col in df.columns if col.startswith(f"{field_id}.")]
            if matching_cols:
                print(f"  Field {field_id} ({description}):")
                print(f"    Found in {name}: {', '.join(matching_cols[:3])}")
                if len(matching_cols) > 3:
                    print(f"    ... and {len(matching_cols) - 3} more")
                found = True
                break
        if not found:
            print(f"  Field {field_id} ({description}): Not found in loaded files")

## 8. Tips and Best Practices

### Performance Tips
- **Start small**: Use `sample_rows` parameter when loading data for initial exploration
- **Load selectively**: Only load files you need using `file_pattern`
- **Use Polars**: The module uses Polars for efficient data handling
- **Memory management**: For full datasets, consider processing files individually

### Data Quality
- **Remember**: This is synthetic data and may not be internally consistent
- **Verify downloads**: MD5 checksums are automatically verified
- **Check field definitions**: Use UK Biobank Showcase to understand field meanings

### Common Use Cases
1. **Phenotype analysis**: Use tabular data for participant characteristics
2. **Clinical history**: Use medical records for diagnosis codes and visit data
3. **Genetic studies**: Use genetic data for GWAS and polygenic risk scores
4. **System testing**: Use bulk files for testing data processing pipelines

In [ ]:
# Example: Efficient processing workflow
print("Recommended workflow for large-scale analysis:")
print("=" * 60)
print(f"""
1. Download only the files you need (saved to {get_cache_dir()} by default):
   download_file("dates_death.tsv", category="tabular")
   download_file("real_fields1.tsv", category="tabular")

2. Load files individually or with patterns (loads from cache by default):
   death_data = load_tabular_data(file_pattern="dates_death")
   measurements = load_tabular_data(file_pattern="real_fields1")

3. Process in chunks if memory is limited:
   # Use Polars lazy evaluation
   df = pl.scan_csv("large_file.tsv", separator="\\t")
   result = df.filter(...).group_by(...).agg(...).collect()

4. Save intermediate results:
   processed_data.write_parquet("processed.parquet")
""")

## 9. Summary

This tutorial covered:
- ✓ Listing available files in the dataset
- ✓ Downloading tabular, medical, and genetic data
- ✓ Loading data into Polars DataFrames
- ✓ Basic data exploration and analysis
- ✓ Merging data from multiple files
- ✓ Understanding UK Biobank field naming conventions

## Additional Resources

- **Official Documentation**: https://biobank.ndph.ox.ac.uk/synthetic_dataset/
- **UK Biobank Showcase** (for field definitions): http://biobank.ndph.ox.ac.uk/showcase/schema.cgi
- **Read Code Mappings**: https://biobank.ctsu.ox.ac.uk/crystal/refer.cgi?id=592

## Next Steps

1. Download the tabular data you need for your analysis
2. Explore the field definitions in UK Biobank Showcase
3. Load and process the data using Polars
4. Perform your analysis while remembering this is synthetic data

Happy analyzing! 🎉